In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv("Dataset1.csv")

# Sort stations
df = df.sort_values(["Train_No", "SN"])


# Convert time columns
df["Arrival_time"] = pd.to_datetime(
    df["Arrival_time"],
    format="%H:%M:%S"
)

df["Departure_Time"] = pd.to_datetime(
    df["Departure_Time"],
    format="%H:%M:%S"
)


# Create train-wise data

train_data = df.groupby("Train_No").agg(
    Total_Distance=("Distance", "max"),
    Start_Time=("Departure_Time", "first"),
    End_Time=("Arrival_time", "last")
).reset_index()


# Handle overnight journeys

train_data.loc[
    train_data["End_Time"] < train_data["Start_Time"],
    "End_Time"
] += pd.Timedelta(days=1)


# Calculate journey duration

train_data["Journey_Duration_Hours"] = (
    train_data["End_Time"] - train_data["Start_Time"]
).dt.total_seconds() / 3600


# Calculate relationship

correlation = train_data[
    ["Total_Distance", "Journey_Duration_Hours"]
].corr()


print("Distance and Journey Duration Data:")

print(
    train_data[
        ["Train_No","Total_Distance","Journey_Duration_Hours"]
    ].head(10)
)


print("\nCorrelation:")
print(correlation)

Distance and Journey Duration Data:
   Train_No  Total_Distance  Journey_Duration_Hours
0       107              78                1.750000
1       108              83                1.916667
2       128             978               22.083333
3       290            2694                8.000000
4       401            1618               12.500000
5       421            1276                9.000000
6       422            1277                2.750000
7       477            2616                3.166667
8       502            1206               23.000000
9       504            1313                1.000000

Correlation:
                        Total_Distance  Journey_Duration_Hours
Total_Distance                  1.0000                  0.6357
Journey_Duration_Hours          0.6357                  1.0000
